<a href="https://colab.research.google.com/github/abeersafi/covid-pneumonia-xray-classification/blob/main/covid_pneumonia_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

API TOKEN: KGAT_281a8fb29ada4037e12396182837867d

CELL 1

In [1]:
import os
from google.colab import drive

# 1. Mount Google Drive for saving model checkpoints later
drive.mount('/content/drive')

# 2. Authenticate using your new Kaggle token
os.environ['KAGGLE_API_TOKEN'] = "KGAT_281a8fb29ada4037e12396182837867d"

# 3. Install Kaggle's new download library
!pip install -q kagglehub

# 4. Download the dataset using kagglehub
import kagglehub
print("Downloading dataset from Kaggle...")
cached_path = kagglehub.dataset_download("anasmohammedtahir/covidqu")

# 5. Move the downloaded files to /content/dataset/ so it matches Cell 2
!mkdir -p /content/dataset/
!cp -r {cached_path}/* /content/dataset/

print("Dataset downloaded and extracted successfully to /content/dataset/!")

Mounted at /content/drive


100%|██████████| 1.15G/1.15G [00:13<00:00, 92.0MB/s]

Extracting files...


Dataset downloaded and extracted successfully to /content/dataset/!


CELL 2

In [2]:
import os
import numpy as np
from PIL import Image
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split

# ---------------------------------------------------------
# 1. Custom Dataset Class with Masking
# ---------------------------------------------------------
class COVID_QU_Dataset(Dataset):
    def __init__(self, image_paths, mask_paths, labels, transform=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]
        label = self.labels[idx]

        # Load image and mask in grayscale
        image = Image.open(img_path).convert("L")
        mask = Image.open(mask_path).convert("L")

        # Extract ROI using numpy
        img_arr = np.array(image)
        mask_arr = np.array(mask)
        binary_mask = (mask_arr > 128).astype(np.uint8)
        roi_arr = img_arr * binary_mask

        # Convert back to RGB for ResNet-50
        roi_image = Image.fromarray(roi_arr).convert("RGB")

        if self.transform:
            roi_image = self.transform(roi_image)

        return roi_image, torch.tensor(label, dtype=torch.long)

# ---------------------------------------------------------
# 2. Transformations (Augmentation for Train only)
# ---------------------------------------------------------
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ---------------------------------------------------------
# 3. Actual Data Loading (Robust Directory Scanner)
# ---------------------------------------------------------
import os

image_paths = []
mask_paths = []
labels = []

class_mapping = {
    'Normal': 0,
    'Pneumonia': 1,
    'COVID-19': 2
}

print("Scanning directory for images and masks...")

# Walk through all directories in the extracted dataset
for root, dirs, files in os.walk('/content/dataset/'):

    # We specifically want the Lung data to extract the lung ROI.
    # We use .lower() to ignore any capitalization/underscore differences caused by extraction.
    if 'lung' in root.lower() and os.path.basename(root).lower() == 'images':

        # Determine the class based on the folder path
        label_idx = None
        for class_name, idx in class_mapping.items():
            if class_name.lower() in root.lower():
                label_idx = idx
                break

        if label_idx is not None:
            # Find the corresponding mask directory (sibling to 'images')
            parent_dir = os.path.dirname(root)
            sibling_dirs = os.listdir(parent_dir)

            mask_dir_path = None
            for d in sibling_dirs:
                if 'mask' in d.lower(): # Catches 'masks', 'Lung Masks', etc.
                    mask_dir_path = os.path.join(parent_dir, d)
                    break

            if mask_dir_path:
                # Pair the images with their masks
                for file in files:
                    if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                        img_path = os.path.join(root, file)
                        mask_path = os.path.join(mask_dir_path, file)

                        # Verify the mask physically exists to prevent loading errors
                        if os.path.exists(mask_path):
                            image_paths.append(img_path)
                            mask_paths.append(mask_path)
                            labels.append(label_idx)

print(f"Successfully loaded {len(image_paths)} valid image-mask pairs.")

# Initialize the dataset with the newly populated lists
full_dataset = COVID_QU_Dataset(image_paths, mask_paths, labels)

# ---------------------------------------------------------
# 4. Partitioning & DataLoaders (Colab Optimized)
# ---------------------------------------------------------
total_size = len(full_dataset)
train_size = int(0.70 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_ds, val_ds, test_ds = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Apply specific transforms
train_ds.dataset.transform = train_transforms
val_ds.dataset.transform = val_test_transforms
test_ds.dataset.transform = val_test_transforms

batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
print("DataLoaders built and ready.")

Scanning directory for images and masks...
Successfully loaded 22657 valid image-mask pairs.
DataLoaders built and ready.


CELL 3

In [ ]:
# ---------------------------------------------------------
# 1. Model Setup: Pre-trained ResNet-50
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze early layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer for 3-class classification
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 3) # SoftMax is handled internally by CrossEntropyLoss
model = model.to(device)

# ---------------------------------------------------------
# 2. Optimization
# ---------------------------------------------------------
criterion = nn.CrossEntropyLoss()
# Only pass the parameters of the newly initialized fc layer to SGD
optimizer = optim.SGD(model.fc.parameters(), lr=0.001, momentum=0.9)

# ---------------------------------------------------------
# 3. The Training Loop
# ---------------------------------------------------------
epochs = 10
save_dir = '/content/drive/MyDrive/BME3180_Project/' # Ensure this folder exists in your Drive!
os.makedirs(save_dir, exist_ok=True)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Training Phase
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    print(f"Epoch [{epoch+1}/{epochs}] | "
          f"Train Loss: {running_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.2f}%")

    # Save checkpoint to Google Drive
    save_path = os.path.join(save_dir, f'resnet50_epoch_{epoch+1}.pth')
    torch.save(model.state_dict(), save_path)

print("Training Complete!")

Training on device: cpu
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 92.0MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


CELL 4

In [ ]:
# ---------------------------------------------------------
# Phase 1: Fine-Tuning the ResNet-50 Higher Layers
# ---------------------------------------------------------
print("Unfreezing layer4 for fine-tuning...")

# 1. Unfreeze the highest convolutional block
for param in model.layer4.parameters():
    param.requires_grad = True

# 2. Re-initialize the SGD optimizer with a LOWER learning rate (1e-4)
# We only pass the parameters that require gradients (layer4 + fc layer)
optimizer_ft = optim.SGD(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, momentum=0.9)
criterion = nn.CrossEntropyLoss()

# 3. Fine-tuning Training Loop (5 Epochs is usually enough for this step)
ft_epochs = 5
for epoch in range(ft_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer_ft.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_ft.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    # Quick Validation Check
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total
    print(f"Fine-Tune Epoch [{epoch+1}/{ft_epochs}] | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

print("Fine-tuning complete. Model is ready for final testing.")

CELL 5

In [ ]:
import numpy as np
# ---------------------------------------------------------
# Phase 2: Final Evaluation on the Unseen Test Set
# ---------------------------------------------------------
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import torch.nn.functional as F

model.eval()
all_preds = []
all_labels = []
all_probs = []

print("Evaluating model on the Test Set. This may take a moment...")

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        # Get raw logits and apply softmax for probabilities
        outputs = model(inputs)
        probabilities = F.softmax(outputs, dim=1)

        # Get final predictions
        _, predicted = torch.max(outputs, 1)

        # Store for scikit-learn
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probabilities.cpu().numpy())

# 1. Classification Report (Precision, Recall, F1-Score, Accuracy)
# Dynamically get unique labels from the test set
unique_labels = np.unique(all_labels)
original_target_names = ['Normal', 'Pneumonia', 'COVID-19']

# Filter target_names to only include those present in the test set
filtered_target_names = [original_target_names[i] for i in unique_labels]

print("\n--- Classification Report ---")
# The 'labels' parameter tells classification_report which labels to consider.
# This prevents errors if not all labels are present in y_true or y_pred.
print(classification_report(all_labels, all_preds, labels=unique_labels, target_names=filtered_target_names))

# 2. ROC-AUC Score (Multi-class requires OVR - One vs Rest)
# Convert all_probs to a numpy array for easier column selection
all_probs_np = np.array(all_probs)

# Select only the columns from all_probs that correspond to the unique labels present in the test set.
# This aligns the number of columns in y_score with the number of classes implied by y_true.
filtered_all_probs = all_probs_np[:, unique_labels]

# Use the unique_labels explicitly so roc_auc_score knows the true classes.
roc_auc = roc_auc_score(all_labels, filtered_all_probs, multi_class='ovr', average='weighted', labels=unique_labels)
print(f"--- Weighted ROC-AUC Score: {roc_auc:.4f} ---")

CELL 6

In [ ]:
# ---------------------------------------------------------
# Phase 3: Grad-CAM Explainability
# ---------------------------------------------------------
# Install the necessary library (runs quietly)
!pip install -q grad-cam

import matplotlib.pyplot as plt
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# 1. Setup Grad-CAM pointing to the highest convolutional layer
target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

# 2. Grab a single batch from the test loader
dataiter = iter(test_loader)
images, labels = next(dataiter)

# Select the first image in the batch
input_tensor = images[0].unsqueeze(0).to(device)
true_label = labels[0].item()

# 3. Generate the Heatmap
# We target the actual true label to see what features support it
targets = [ClassifierOutputTarget(true_label)]
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]

# 4. Format the image for Matplotlib (convert from PyTorch Tensor)
# Un-normalize the image back to visual range [0, 1]
img_for_plot = images[0].permute(1, 2, 0).numpy()
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
img_for_plot = std * img_for_plot + mean
img_for_plot = np.clip(img_for_plot, 0, 1)

# 5. Overlay heatmap on original image
visualization = show_cam_on_image(img_for_plot, grayscale_cam, use_rgb=True)

# 6. Plot the results
class_names = ['Normal', 'Non-COVID', 'COVID-19']
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img_for_plot)
plt.title(f"Original ROI (True Class: {class_names[true_label]})")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(visualization)
plt.title("Grad-CAM Heatmap")
plt.axis('off')

plt.show()

CELL 7

In [ ]:
# ---------------------------------------------------------
# Phase 4: Confusion Matrix Visualization
# ---------------------------------------------------------
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

# 1. Generate the raw confusion matrix from Cell 5's data
cm = confusion_matrix(all_labels, all_preds)

# 2. Define the class names in the correct order
class_names = ['Normal', 'Pneumonia', 'COVID-19']

# 3. Plot it using Seaborn for a clean, professional look
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names,
            annot_kws={"size": 14}) # Make the numbers easy to read

# 4. Add labels and title
plt.ylabel('Actual (True) Diagnosis', fontsize=12, fontweight='bold')
plt.xlabel('Model Prediction', fontsize=12, fontweight='bold')
plt.title('ResNet-50 Multi-Class Confusion Matrix', fontsize=14, pad=15)

# 5. Display the plot
plt.tight_layout()
plt.show()